# Dim Localizacao

In [0]:
import pyspark.sql.functions as sf

In [0]:
spark.sql('USE CATALOG saude_sus')

In [0]:
df_tru = spark.read.table('saude_sus.trusted.tru_estabelecimento')

In [0]:
df_dim_loc = df_tru.select(
    sf.md5(sf.concat_ws("-", 
        sf.col("COD_UF"), 
        sf.col("COD_MUNICIPIO"), 
        sf.upper(
            sf.coalesce(
                sf.trim(
                    sf.upper(
                        sf.col("END_BAIRRO_ESTABELECIMENTO")
                    )
                ), 
                sf.lit("NAO INFORMADO")
            )
        ),
        sf.upper(
            sf.coalesce(
                sf.trim(
                    sf.upper(
                        sf.col("END_CEP_ESTABELECIMENTO")
                    )
                ), 
                sf.lit("NAO INFORMADO")
            )
        ),
        sf.upper(
            sf.coalesce(
                sf.trim(
                    sf.upper(
                        sf.col("END_LOGRADOURO_ESTABELECIMENTO")
                    )
                ), 
                sf.lit("NAO INFORMADO")
            )
        )
    )).alias("SK_LOCALIZACAO"),

    sf.col("COD_UF"),
    sf.col("DSC_UF"),
    sf.col("COD_MUNICIPIO"),
    sf.upper(sf.col("END_BAIRRO_ESTABELECIMENTO")).alias("END_BAIRRO_ESTABELECIMENTO"),
    sf.col('END_CEP_ESTABELECIMENTO'),
    sf.upper(sf.col('END_LOGRADOURO_ESTABELECIMENTO')).alias("END_LOGRADOURO_ESTABELECIMENTO"),
    sf.col('END_LATITUDE_ESTABELECIMENTO'),
    sf.col('END_LONGITUDE_ESTABELECIMENTO')
)

In [0]:
try:
    df_dim_existente = spark.table("saude_sus.refined.dim_localizacao").select("SK_LOCALIZACAO")
except:
    df_dim_existente = spark.createDataFrame([], df_dim_loc.select("SK_LOCALIZACAO").schema)

df_para_inserir = df_dim_loc.join(
    df_dim_existente, 
    on="SK_LOCALIZACAO", 
    how="left_anti"
).distinct()

df_para_inserir.write.format("delta").mode("append").saveAsTable("saude_sus.refined.dim_localizacao")